# Selene-Core Advanced: Minimal C Plugin + Python Wrapper

This notebook demonstrates a compact end-to-end plugin workflow:
1. Write a tiny simulator plugin in C
2. Compile it to a shared library via `python -m ziglang cc`
3. Wrap it with a Python `selene_core.Simulator` class
4. Use it in `selene_sim.build(...).run_shots(...)`

The plugin is intentionally simple and not physically accurate. It is a skeleton for experts to extend.

In [ ]:
import os
import sys
import subprocess
import tempfile
from pathlib import Path
from dataclasses import dataclass

from guppylang import guppy
from guppylang.std.quantum import qubit, h, measure
from guppylang.std.builtins import result

from selene_core import Simulator, get_include_directory
from selene_sim.build import build

## 1) Write C plugin source

This simulator returns pseudo-random measurement bits with configurable `--bias=<float>`.

In [ ]:
work = Path(tempfile.mkdtemp(prefix="selene_c_plugin_"))
src = work / "c_minimal_simulator.c"

c_code = r'''
#include <selene/simulator.h>
#include <stdint.h>
#include <stdbool.h>
#include <stdlib.h>
#include <string.h>
#include <stdio.h>

typedef struct PluginState {
    uint64_t n_qubits;
    uint64_t rng_state;
    double bias;
} PluginState;

static uint64_t xorshift64(uint64_t *state) {
    uint64_t x = *state;
    x ^= x << 13;
    x ^= x >> 7;
    x ^= x << 17;
    *state = x;
    return x;
}

static int in_bounds(PluginState *s, uint64_t q) {
    return q < s->n_qubits;
}

uint64_t selene_simulator_get_api_version(void) {
    // reserved=0, major=0, minor=1, patch=0
    return ((uint64_t)0 << 24) | ((uint64_t)0 << 16) | ((uint64_t)1 << 8) | (uint64_t)0;
}

int32_t selene_simulator_init(SeleneSimulatorInstance *instance, uint64_t n_qubits, uint32_t argc, const char *const *argv) {
    PluginState *state = (PluginState *)calloc(1, sizeof(PluginState));
    if (!state) return -1;
    state->n_qubits = n_qubits;
    state->rng_state = 0x123456789abcdefULL;
    state->bias = 0.5;

    for (uint32_t i = 0; i < argc; ++i) {
        const char *arg = argv[i];
        if (strncmp(arg, "--bias=", 7) == 0) {
            state->bias = atof(arg + 7);
        }
    }

    if (state->bias < 0.0 || state->bias > 1.0) {
        free(state);
        return -2;
    }

    *instance = (SeleneSimulatorInstance)state;
    return 0;
}

int32_t selene_simulator_exit(SeleneSimulatorInstance instance) {
    free((PluginState *)instance);
    return 0;
}

int32_t selene_simulator_shot_start(SeleneSimulatorInstance instance, uint64_t shot_id, uint64_t seed) {
    PluginState *s = (PluginState *)instance;
    s->rng_state = seed ^ (shot_id + 0x9e3779b97f4a7c15ULL);
    return 0;
}

int32_t selene_simulator_shot_end(SeleneSimulatorInstance instance) {
    (void)instance;
    return 0;
}

int32_t selene_simulator_operation_rxy(SeleneSimulatorInstance instance, uint64_t qubit, double theta, double phi) {
    (void)theta; (void)phi;
    PluginState *s = (PluginState *)instance;
    return in_bounds(s, qubit) ? 0 : -10;
}

int32_t selene_simulator_operation_rzz(SeleneSimulatorInstance instance, uint64_t q0, uint64_t q1, double theta) {
    (void)theta;
    PluginState *s = (PluginState *)instance;
    return (in_bounds(s, q0) && in_bounds(s, q1)) ? 0 : -11;
}

int32_t selene_simulator_operation_rz(SeleneSimulatorInstance instance, uint64_t qubit, double theta) {
    (void)theta;
    PluginState *s = (PluginState *)instance;
    return in_bounds(s, qubit) ? 0 : -12;
}

int32_t selene_simulator_operation_measure(SeleneSimulatorInstance instance, uint64_t qubit) {
    PluginState *s = (PluginState *)instance;
    if (!in_bounds(s, qubit)) return -13;

    uint64_t r = xorshift64(&s->rng_state);
    double u = (double)(r & 0xFFFFFFFFULL) / (double)0x100000000ULL;
    return (u < s->bias) ? 1 : 0;
}

int32_t selene_simulator_operation_postselect(SeleneSimulatorInstance instance, uint64_t qubit, bool target_value) {
    (void)target_value;
    PluginState *s = (PluginState *)instance;
    return in_bounds(s, qubit) ? 0 : -14;
}

int32_t selene_simulator_operation_reset(SeleneSimulatorInstance instance, uint64_t qubit) {
    PluginState *s = (PluginState *)instance;
    return in_bounds(s, qubit) ? 0 : -15;
}

int32_t selene_simulator_get_metrics(SeleneSimulatorInstance instance, uint8_t nth_metric, char *tag_ptr, uint8_t *datatype_ptr, uint64_t *data_ptr) {
    PluginState *s = (PluginState *)instance;
    if (nth_metric == 0) {
        strcpy(tag_ptr, "configured_bias");
        *datatype_ptr = 3; // f64
        memcpy(data_ptr, &s->bias, sizeof(double));
        return 0;
    }
    return 1;
}

int32_t selene_simulator_dump_state(SeleneSimulatorInstance instance, const char *file, const uint64_t *qubits, uint64_t n_qubits) {
    (void)instance; (void)qubits;
    FILE *f = fopen(file, "w");
    if (!f) return -16;
    fprintf(f, "state dump not implemented; requested %llu qubits\n", (unsigned long long)n_qubits);
    fclose(f);
    return 0;
}
'''

src.write_text(c_code)
src

## 2) Compile shared library with `ziglang`

In [ ]:
include_dir = get_include_directory()
lib = work / "libselene_c_minimal_simulator.so"

cmd = [
    sys.executable,
    "-m",
    "ziglang",
    "cc",
    "-shared",
    "-fPIC",
    str(src),
    "-I",
    str(include_dir),
    "-o",
    str(lib),
]

print("Running:", " \
,
,

: 
,
: {
: 

: [
3
,
,

: 
,
: {
: 

: [
,

    @property
    def library_file(self) -> Path:
        return lib

    def get_init_args(self) -> list[str]:
        return [f"--bias={self.bias}"]

## 4) Use the plugin with a simple Guppy program

In [ ]:
@guppy
def main() -> None:
    q0 = qubit()
    h(q0)
    result("c0", measure(q0))

runner = build(main.compile(), name="c_plugin_demo")
shots = [dict(shot) for shot in runner.run_shots(
    simulator=CMinimalSimulator(bias=0.8, random_seed=123),
    n_qubits=1,
    n_shots=20,
)]

n_true = sum(1 for shot in shots if shot["c0"])
print("shots:", shots)
print("true count:", n_true, "/", len(shots))